# TVP-SV-VAR And Robustness

This notebook focuses on the time-varying scenario evidence and the main robustness checks. It is designed to load the current outputs by default, with an optional rerun cell if you want to refresh the TVP-SV-VAR and robustness artifacts from the notebook.


In [ ]:
from __future__ import annotations

from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / "pyproject.toml").exists():
    if PROJECT_ROOT.parent == PROJECT_ROOT:
        raise RuntimeError("Could not locate the project root containing pyproject.toml.")
    PROJECT_ROOT = PROJECT_ROOT.parent

SRC_PATH = PROJECT_ROOT / "src"
if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import Image, display

pd.options.display.max_columns = 50
pd.options.display.float_format = lambda value: f"{value:,.3f}"
plt.style.use("seaborn-v0_8-whitegrid")

from laos_fx_oil_macro.notebook_helpers import (
    build_notebook_project,
    compact_interval_pivot,
    load_csv,
    plot_tvp_irf_scenarios,
    round_numeric_columns,
    run_pipeline_from_notebook,
)

project = build_notebook_project(PROJECT_ROOT)
EVENT_DATES = ["2022-03-31", "2023-10-31", "2026-02-28"]
project.root


In [ ]:
REFRESH_TVP = False
TVP_DRAWS = 1000
TVP_BURNIN = 300
TVP_THIN = 2
TVP_HORIZONS = 12
REFRESH_TVP


In [ ]:
if REFRESH_TVP:
    tvp_manifest = run_pipeline_from_notebook(
        project,
        tvp_draws=TVP_DRAWS,
        tvp_burnin=TVP_BURNIN,
        tvp_thin=TVP_THIN,
        tvp_horizons=TVP_HORIZONS,
        skip_tvp=False,
    )
    display(
        pd.DataFrame(
            {
                "artifact": tvp_manifest.keys(),
                "path": [str(path.relative_to(project.root)) for path in tvp_manifest.values()],
            }
        )
    )
else:
    print("REFRESH_TVP is False. Loading the current TVP-SV-VAR and robustness outputs from disk.")


In [ ]:
required_paths = {
    "irf": project.models_dir / "tvp_svar_irf_summary.csv",
    "fit": project.models_dir / "tvp_svar_fit_summary.csv",
    "robustness": project.tables_dir / "robustness_summary.csv",
    "gpr_to_oil": project.tables_dir / "gpr_to_oil_hac_ols.csv",
    "gpr_alt_to_oil": project.tables_dir / "gpr_alt_to_oil_hac_ols.csv",
    "lp_fx_to_cpi_inf": project.tables_dir / "lp_fx_to_cpi_inf.csv",
    "lp_fx_to_inflation": project.tables_dir / "lp_fx_to_inflation.csv",
    "lp_fx_to_inflation_truncated": project.tables_dir / "lp_fx_to_inflation_truncated.csv",
}
missing = [str(path.relative_to(project.root)) for path in required_paths.values() if not path.exists()]
if missing:
    raise FileNotFoundError("Missing TVP-SV-VAR or robustness outputs: " + ", ".join(missing))

irf = load_csv(required_paths["irf"])
fit = load_csv(required_paths["fit"])
robustness = load_csv(required_paths["robustness"])
gpr_to_oil = load_csv(required_paths["gpr_to_oil"])
gpr_alt_to_oil = load_csv(required_paths["gpr_alt_to_oil"])
lp_fx_to_cpi_inf = load_csv(required_paths["lp_fx_to_cpi_inf"])
lp_fx_to_inflation = load_csv(required_paths["lp_fx_to_inflation"])
lp_fx_to_inflation_truncated = load_csv(required_paths["lp_fx_to_inflation_truncated"])


## TVP-SV-VAR run summary and impact responses


In [ ]:
display(round_numeric_columns(fit))
scenario_h0 = irf.loc[irf["horizon"] == 0, ["date", "impulse", "response", "median", "lower_90", "upper_90"]]
display(round_numeric_columns(scenario_h0))


The scenario summaries show the strongest oil-to-FX and oil-to-inflation responses around March 2022, smaller but still positive responses around October 2023, and much weaker responses by February 2026. The time-varying model therefore supports the state-dependent amplification story, but it also shows that the stressed regime is not flat over time.


## Scenario IRFs: `oil_shock -> fx_dep`


In [ ]:
fig = plot_tvp_irf_scenarios(irf, impulse="oil_shock", response="fx_dep", dates=EVENT_DATES)
display(fig)
plt.close(fig)


## Scenario IRFs: `oil_shock -> inflation_mom`


In [ ]:
fig = plot_tvp_irf_scenarios(irf, impulse="oil_shock", response="inflation_mom", dates=EVENT_DATES)
display(fig)
plt.close(fig)


## Scenario IRFs: `fx_dep -> inflation_mom`


In [ ]:
fig = plot_tvp_irf_scenarios(irf, impulse="fx_dep", response="inflation_mom", dates=EVENT_DATES)
display(fig)
plt.close(fig)


These scenario IRFs support the main narrative, but they should not be oversold as a sharp geopolitical-conflict identification exercise. The defensible claim is narrower: when Laos was under severe stress, the oil and FX transmission block was much more inflationary than in calmer periods.


## Robustness: `fx_dep -> CPI_Inf`


In [ ]:
display(compact_interval_pivot(lp_fx_to_cpi_inf, index="horizon", columns="regime"))


Using year-over-year inflation makes the response more gradual, but the post-2022 pass-through remains materially stronger than before the break. That supports the interpretation that the result is not an artifact of the monthly inflation transformation alone.


## Robustness: truncated sample check


In [ ]:
baseline_h0 = lp_fx_to_inflation.loc[lp_fx_to_inflation["horizon"] == 0, ["regime", "coefficient", "lower_90", "upper_90"]].copy()
baseline_h0["specification"] = "baseline"
truncated_h0 = lp_fx_to_inflation_truncated.loc[
    lp_fx_to_inflation_truncated["horizon"] == 0,
    ["regime", "coefficient", "lower_90", "upper_90"],
].copy()
truncated_h0["specification"] = "truncated_at_2025_12"
truncated_compare = pd.concat([baseline_h0, truncated_h0], ignore_index=True)
display(round_numeric_columns(truncated_compare[["specification", "regime", "coefficient", "lower_90", "upper_90"]]))


Truncating the sample at December 2025 leaves the main post-2022 pass-through result intact. That makes the benchmark conclusion harder to dismiss as an endpoint artifact.


## Robustness: alternative GPR check


In [ ]:
alt_gpr_terms = pd.concat(
    [
        gpr_to_oil.loc[gpr_to_oil["term"] == "GPR_ME", ["regime", "term", "coefficient", "p_value"]],
        gpr_alt_to_oil.loc[gpr_alt_to_oil["term"] == "GPR_ME_alt", ["regime", "term", "coefficient", "p_value"]],
    ],
    ignore_index=True,
)
display(round_numeric_columns(alt_gpr_terms))


The alternative GPR construction does not materially strengthen the monthly `GPR -> oil` relation. That is useful because it means the paper's contribution should stay centered on exchange-rate amplification, not on a fragile monthly geopolitical-risk coefficient.


## What is supported, and what should not be oversold


In [ ]:
gpr_scenario_h0 = irf.loc[
    (irf["horizon"] == 0) & (irf["impulse"] == "GPR_ME") & (irf["response"] == "oil_shock"),
    ["date", "median", "lower_90", "upper_90"],
]
display(round_numeric_columns(gpr_scenario_h0))
display(round_numeric_columns(robustness))


The evidence supports a stronger post-2022 FX pass-through mechanism and time-varying amplification in the stressed period. What it does **not** support is a strong direct reduced-form monthly `GPR -> oil` link in this sample. Keep the February 2026 date framed as a scenario IRF evaluated at the sample endpoint, not as realized ex post evidence.
